# Using the CVXOPT Package to solve SVM Problems

CVXOPT is a free solver system for constrained optimistion problems (http://cvxopt.org) with an interface written in Python.

The interface is similar to that provided by NumPy, the only issue being that CVXOPT has its own notion of matrices and vectors, so programs using CVXOPT have to employ these CVXOPT versions.  Luckily there is a good interface to NumPy, so NumPy <font color="#2020ff"><i>array</i></font> objects, along with standard Python lists, can easily be translated to and from CVXOPT matrices.

The most important CVXOPT imports are "<font color="#2020ff"><b>matrix</b></font>", for matrix construction, and "<font color="#2020ff"><b>solvers</b></font>".

The CVXOPT "<font color="#2020ff"><b>mul</b></font>" function is useful if you want to do element-by-element (i.e., Hadamard) multiplication of matrices (we'll use this below).

In [ ]:
import cvxopt as cvx
import numpy as np

Use "<font color="#2020ff"><b>matrix</b></font>" to build a matrix.  The only issue is that the arguments are in <font color="#ff2040"><em>column-major</em></font> order (the opposite to the NumPy convention).  To create a double-precision (i.e., using 64-bit numbers) float matrix, make sure that at least one of the entries has a decimal point (or include the keyword argument <font color="#2020ff">tc='d'</font>).

Not that CVXOPT expects all numeric arguments to its routines to be double-precision, either as "<font color="#2020ff"><b>matrix</b></font>" objects or scalars.

In [ ]:
A = cvx.matrix([[2.,0,3],[0,1,4]])   # Creates a 3 row, 2 column matrix (column-major)

B = np.array([[2.,0,3],[0,1,4]])     # NumPy array has 2 rows and 3 columns (row-major).

Examining $A$, we see that it has $3$ rows and $2$ columns, i.e., it was constructed with $[2,0,3]$ down its first column and $[0,1,4]$ down its second.  The NumPy array $B$, on the other hand, was constructed with $[2,0,3]$ along its first <em>row</em>, etc.

In [ ]:
A                                    # Just gives shape and type, not contents.

In [ ]:
print(A)                             # Explicit print needed to view contents. Note column-major.

In [ ]:
print(f"{B}\n\n{B.shape}")           # Note difference to NumPy array (row-major).

In [ ]:
print(cvx.matrix(5.0))               # A 1x1 matrix can be created using a single scalar

In CVXOPT, vectors are just $N \times 1$ matrices, i.e., column vectors by default.  Notice that they are really 2-d objects, with $N$ rows and a single column (unlike NumPy vectors, which are purely 1-d).

In [ ]:
vector = cvx.matrix([1,1,1,1], tc='d')

In [ ]:
vector                               # No separate "vector", vectors are matrices (unlike NumPy).

In [ ]:
print(vector)

Matrix multiplication in CVXOPT overloads the "<font color="#2020ff"><b>*</b></font>" operator, not "<font color="#2020ff"><b>@</b></font>".  

Here we have
$$
\left[\begin{array}{cc} 2 & 0 \\ 0 & 1 \\ 3 & 4 \end{array}\right]
\left[\begin{array}{c} 5 \\ 10  \end{array}\right] =
\left[\begin{array}{c} 10 \\ 10 \\ 55 \end{array}\right]
$$

(Note that CVXOPT consistently uses scientific notation in its printed outputs.)

In [ ]:
vec = cvx.matrix([5.,10])

print(A)
print(vec)

print(A*vec)                         # '*' is matrix multiplication in CVXOPT, not "@".

The easiest way to make a row vector is simply to transpose a column vector.  CVXOPT has the "<font color="#2020ff">.T</font>" method, which works much like its NumPy equivalent, but will properly transpose CVXOPT vectors.

In [ ]:
print(vec.T)                         # To make a row vector, simply transpose a column vector.

In [ ]:
vec.T                                # Shape of transpose: 1 row, 2 columns.

In [ ]:
print(cvx.matrix([[1],[2]]))         # Or you can do this!  Not recommended.

### Using the 'qp' solver.

Here's a manual run-through of building a linear SVM to solve the 2-input OR problem.

The linear SVM is a set of Lagrange multipliers $\vec{\lambda}$ that maximises the Wolfe Dual function

$$
W(\vec{\lambda}) = \left(\sum_{i=1}^{n} \lambda_i\right) - \left({\textstyle \frac 1 2}
\sum_{i=1}^{n} \sum_{j=1}^{n} \lambda_i \lambda_j
t_i t_j \: \vec{x}_i . \vec{x}_j \right)
$$

subject to $\lambda_i \geq 0$ and $\sum_i t_i \lambda_i = 0$.

In CVXOPT the "<font color="#2020ff"><b>qp</b></font>" solver wants to <i>minimize</i>

$$
Q(\vec{\lambda}) = {\textstyle \frac 1 2} \vec{\lambda}^t {\mathbf P} \vec{\lambda} + \vec{q}^t \vec{\lambda}
$$

subject to ${\mathbf G} \vec{\lambda} \leq \vec{h}$ and ${\mathbf A} \vec{\lambda} = \vec{b}$.

So we have to recast the problem and constraints into the form of appropriate matrices and vectors.

First create a vector of the desired outputs $\vec{t}$.  This plays the role of the ${\mathbf A}$ 
matrix in the above, and is also used in the construction of the ${\mathbf P}$ 
matrix.  

Note that the vector is forced to be a
float one because of the decimal point in the first entry.  Alternatively, specify <font color="#2020ff">tc='d'</font> as a keyword in the constructor.

It is important to use double-precision float matrices everywhere here, as the "<font color="#2020ff"><b>qp</b></font>" solver expects all its arguments to be matrices of double-precision numbers.

In [ ]:
t=cvx.matrix([-1.,1,1,1])            # Force double-precision by "-1.".

It can be useful to create an outer-product matrix $[t_i t_j]$ of the training inputs and multiply it elementwise by an outer-product matrix of input vectors to create the CVXOPT $P$ matrix.

In [ ]:
print(t*t.T)

Input vectors $\vec{x}_i$, One vector per row of the $X$ matrix (i.e., the matrix is a stack of row-vectors).  Note the use of the <font color="#2020ff">tc='d'</font> keyword to the constructor in order to force double-precision.

In [ ]:
X=cvx.matrix([[0,0],[0,1],[1,0],[1,1]], tc='d').T

In [ ]:
print(X)

Outer-product matrix of input vectors $\vec{x}_i .\vec{x}_j$.  This is the Gram matrix for an identity kernel.

For a kernel
SVM, you would need to build an $N \times N$ matrix of the results of applying the kernel
function, $K(\vec{x}_i.\vec{x}_j)$.

In [ ]:
print(X*X.T)

The $P$ matrix to be input to the "<font color="#2020ff"><b>qp</b></font>" solver is the elementwise (Hadamard) product of these two outer-product matrices.  In CVXOPT, the routine "<font color="#2020ff"><b>mul</b></font>" provides element by element multiplication of two matrices.

In [ ]:
P=cvx.mul(t*t.T,X*X.T)

In [ ]:
print(P)

Now construct all the other required matrices for the solver.

The $\vec{q }$ vector is supplied to the solver as a <i>column</i> vector of $-1$'s with $N$ elements  (the equation for $Q$ shows the use of the transpose of $\vec{q }$, which is a row vector, but the solver expects to get the basic $\vec{q }$, a column vector!).

In [ ]:
q = cvx.matrix(4*[-1.0])      #  4*[-1.0] makes a 4-element Python list containing -1's.

For the hard-margin SVM needed for 2-input OR, the $G$ matrix, needed to handle the constraints $\lambda_i \geq 0$, is a $4 \times 4$ matrix with $-1$'s on its main diagonal and $0$'s elsewhere.  The NumPy "<font color="#2020ff"><b>eye</b></font>" function can help with this.

Note that the constraints $\lambda_i \geq 0$ have to be recast to $-\lambda_i \leq 0$ because the "<font color="#2020ff"><b>qp</b></font>" solver expects its inequality contraints in a matrix-vector inequality $G \vec\lambda \leq \vec{h}$.

In [ ]:
G = cvx.matrix(-np.eye(4))    # Note the minus sign!  Important!

In [ ]:
G

In [ ]:
print(G)

The $\vec{h}$ vector is also needed.  For this hard-margin SVM, it's just a column vector of zeros, so that the constraint equation $G \vec\lambda \leq \vec{h}$ becomes $-\vec\lambda
 \leq 0$ or $\vec\lambda \geq 0$.

In [ ]:
h = cvx.matrix(4*[0.0])

This handles most of the construction of the SVM problem in a form suitable for the solver.  Only one thing remains, the <em>equality</em> constraint that states $\sum t_i \lambda_i = 0$.

The "<font color="#2020ff"><b>qp</b></font>" handles such constraints via the matrix-vector equation $A \vec\lambda \leq \vec{b}$, which is actually more general than we need for the SVM problem.  We can construct $A$ as a 1-row, $n$-column vector containing the elements of $t_i$, i.e, just the transpose of the column-vector $\vec{t}$ that we have created already.

What about $\vec{b}$?  Clearly, in this problem, it's just the scalar $0$.  Routine "<font color="#2020ff"><b>qp</b></font>" wants a vector here however, so we can just pass the singleton matrix "<font color="#2020ff">matrix(0.0)</font>" as the $\vec{b}$ parameter.  Note that for a singleton matrix we can just pass a scalar to the constructor (no square brackets needed - although not illegal).

Here I'll call the $\vec{b}$ vector "<tt>bv</tt>", as I want to keep "<tt>b</tt>" to represent the bias of the SVM.

Finally we can call the solver to find the Lagrange multipliers for the problem.


In [ ]:
bv = cvx.matrix(0.0)

r = cvx.solvers.qp(P,q,G,h,t.T,bv)

If the information printed out by the solver ends with 'Optimal solution found', it has worked and found the Lagrange multipliers for the problem.

The solver returns a lot of data about the solution in a Python dictionary object, here "<font color="#2020ff"><b>r</b></font>".

In [ ]:
r

The Lagrange multiplers are returned as a CVXOPT matrix under the '<font color="#ff2020">x</font>' key in the dictionary.

In [ ]:
print(r['x'])


Note that the slack entry is not exactly zero, instead it is a very small value, close to zero.  This is an artifact of the solver's internal algorithm (a so-called "primal-dual interior-point method"), and is normal behaviour.  However, it is good practice to "clean-up" the results, removing the numerical noise.  

For CVXOPT the solvers return about 6 decimal digits of valid result, with noise beneath this.  So we can round the multipliers to 6 decimal digits using Python's "<font color="#2020ff"><b>round</b></font>" function.


In [ ]:
lambdas = cvx.matrix([round(li,6) for li in r['x']])
print(lambdas)

The solution status is returned as a string under the '<font color="#ff2020">status</font>' key.  String 'optimal' means a valid solution has been found.

In [ ]:
print(r['status'])

It can sometimes be convenient to convert the CVXOPT matrix containing the Lagrange multipliers to a more common Python data structure, such as a list or NumPy array.

In [ ]:
list(lambdas)

In [ ]:
np.array(lambdas)

This representation prints out more significant digits of the solution.  Note that there is still some numerical noise in one of the multipliers, but this is acceptable.

-------------

Once we have the Lagrange multipliers for this (linear) SVM problem, the weight vector is generated from

$$
\vec{w} = \sum_i t_i \lambda_i \vec{x}_i
$$

This can most conveniently be achieved by converting all the matrices and vectors to NumPy, noting that $X$, the matrix of training values, is arranged as a stack of row vectors representing the training inputs.

In [ ]:
w = np.sum(np.array(t) * np.array(lambdas) * np.array(X), axis=0)

In [ ]:
print(w, w.shape)

We get back a NumPy vector $[2,2]$, repesenting the two weights of the system. 

The bias, $b$, can be retreived by solving for any margin point (i.e., non-zero $\lambda_i$). 

$$
    \vec{w}^t\vec{x}_i + b = t_i
$$
(note the use of $t_i$ to get the polarity right).

However, it is numerically better to take an average of the solutions over all the support vectors.

In [ ]:
supports = 0
b = 0.0
for i in range(np.array(X).shape[0]):
    x_i = np.array(X)[i]
    if lambdas[i] > 1e-10:
        print(f"Support vector {x_i}, t_i {t[i]}  result {t[i] - w @ x_i}")
        b += t[i] - w @ x_i
        supports += 1
b /= supports

print(f"\nBias {b} calculated over {supports} support vectors")

------------------------

Let's test the system.  Activations $> 0$ return $+1$ and $\leq 0$ return $-1$.  So we define a rouine to do the classification.

In [ ]:
def classify(xvec, w, b):
    return 1 if w @ xvec + b > 0 else -1

And we do an exhausive test for this problem.  The result is a 2-input OR, inputs binary encoded, output bipolar-binary encoded.

In [ ]:
for x1 in [0,1]:
    for x2 in [0,1]:
        xvec = [x1,x2]
        print(f"{xvec} --> {classify(xvec,w,b):+2d}")


---------------------

### 'qp' used to build 2-input XOR

Let's build another classifier, this one for 2-input XOR.  We can reuse the CVXOPT $X$ matrix of inputs, but we have to redefine the outputs.  We can reuse the $G$ matrix and the $\vec{q}$ and $\vec{h}$ vectors.

Trying the approach used for OR we get.

In [ ]:
t_xor = cvx.matrix([-1.,1,1,-1])            # Note 1. to for floating-point

P_xor_v1 = cvx.mul(t_xor*t_xor.T,X*X.T)     # Building P martix as for OR

r_xor_v1 = cvx.solvers.qp(P_xor_v1,q,G,h,t_xor.T,bv)

This time the solver has failed, as we were using the same approach as for the linear OR problem, but XOR is not linearly-separable.  The result status comes back as 'unknown', the solver has failed.  If we examine the Lagrange multipliers, we see that they have all raced off in lockstep towards $\infty$.

In [ ]:
r_xor_v1['status']

In [ ]:
print(r_xor_v1['x'])  # "Lagrange multiplers" are under key 'x'.

The issue is the nonlinearity of the XOR problem.  We need a <font color="red"><i><b>Nonlinear Kernel</b></i></font> to fix this.  

Let's try the quadratic polynomial kernel 

$$
K(\vec{x}_i,\vec{x}_j) = (\vec{x}_i . \vec{x}_j + 1)^2
$$.

In [ ]:
def K(xi,xj):
    """Accepts CVXOPT row vectors xi, xj, or matrices of such.  
       Generates the polynomial kernel (xi.xj + 1)**2."""
    a = xi * xj.T
    return (a + 1)**2

In [ ]:
x2, x3 = X[2,:], X[3,:]     # Do a test of K using [1,0] and [1,1]
print(x2)
print(x3)
print(K(x2,x3))


We could now use a pair of nested for-loops to calculate all all combinations of $K(\vec{x}_i,\vec{x}_j)$, but since we have all the training vectors as rows in CVXOPT matrix $X$, we can easily calculate all combinations of $K(\vec{x}_i,\vec{x}_j)$ using a <i>single</i> call to $K$.

In [ ]:
print(K(X,X))

Now the $P$ matrix is of the form $[t_i t_j K(\vec{x}_i,\vec{x}_j)]$ for a nonlinear kernel.  We can generate this from

$$
P = \vec{t}\; \vec{t}^T \; K(X,X)
$$

where each element of the square matrix $\vec{t}\; \vec{t}^T$ is multiplied by its corresponding element in the square matrix $K(X,X)$.

In [ ]:
P_xor_v2 = cvx.mul(t_xor * t_xor.T, K(X,X))

In [ ]:
print(P_xor_v2)

In [ ]:
r_xor_v2 = cvx.solvers.qp(P_xor_v2,q,G,h,t_xor.T,bv)

In [ ]:
lambdas_xor =  cvx.matrix([round(li,6) for li in r_xor_v2['x']])
print(lambdas_xor)

All the Lagrange multipliers are non-zero, so there are no slacks in this problem. 

Because of the nonlinear kernel mapping, there is no explicit 2-d weight vector $w$ this time, but we still need a bias.  We use the equation

$$
b = t_s - \sum_{r \in S} \lambda_r t_r K(\vec{x}_r,\vec{x}_s)
$$

for any support vector $s$ and the set of support vectors $S$ (in the sum above we'll just sum over all vectors, as non-support vectors have no effect).  As before, it's numerically more stable to average over all the support vectors:

In [ ]:
def makeB(K,lambdas,X,t,zero_tol=1e-10, precision=6):
    """Make bias under kernel K.  Lambdas and ts cvxopt column vectors.
       X is a cvxopt matrix with 1 training vector per row."""
    supports = 0
    b = 0
    for s in range(len(lambdas)):
        if lambdas[s] > zero_tol:  # lambdas[s] is a support vector if > 0.
            supports += 1
            b += t[s] - sum(cvx.mul(cvx.mul(lambdas,t),K(X,X[s,:])))
    return round(b / supports, precision)

In [ ]:
b_xor = makeB(K,lambdas_xor,X,t_xor)

In [ ]:
b_xor

The classification routine, similarly, can't make use of an explicit weight vector. Instead it needs to use the kernel, along with the Lagrange multiplers and the training outputs in the equation

$$
o(\vec{x}_c) = 
\left\{
\begin{array}{ccl} 
  +1 & & \sum_{r \in S} \lambda_r t_r K(\vec{x}_r,\vec{x}_c) + b > 0 \\
  -1 & & otherwise
\end{array}\right.
$$


There's some messing about at the start of this routine, but that's only to ensure that it can classify vectors represented as CVXOPT row vectors, or, alternatively, simple Python lists or NumPy arrays.  The core of the routine is in the section commented as "Actual calculation".


In [ ]:
def kernClassify(xvec,K,lambdas,Xs,ts,b,zero_tol=1e-10):
    """Requires X to be a matrix of training inputs arranged as cvxopt row vectors.
       'xvec', the input to be classified can be a cvxopt row vector, a Python list
       representing a row vector, or a NumPy 1-d array."""
    # Do conversions on xvec if needed to coerce to a cvxopt matrix with 1 row and n cols
    # (i.e., a cvxopt row vector).
    if isinstance(xvec,list):          # Convert Python list to cvxopt row vector.
        xvec = cvx.matrix(xvec).T
    elif isinstance(xvec,np.ndarray):  # Convert NumPy array to cvxopt row vector.
        xv = xvec
        xvec = cvx.matrix(xv) if xv.shape[0] == 1 else cvx.matrix(xv).T
    #-----------------------------------------------------------
    # Actual calculation.  y is activation level.
    y = b + sum(cvx.mul(cvx.mul(lambdas,ts),K(Xs,xvec)))
    return +1 if y > 0 else -1

In [ ]:
for x1 in [0,1]:
    for x2 in [0,1]:
        xvec = [x1,x2]
        print(f"{xvec} --> "
              f"{kernClassify(xvec, K, lambdas_xor, X, t_xor, b_xor):+2d}")

-------------------

### 3-input XOR

Let's try 3-input XOR.  This time we will find that the quadratic polynomial kernel isn't good enough, but we can try a higher exponent, $n=3$, and find this works very well.


$$K_n(\vec{x}_i,\vec{x}_j) = (\vec{x}_i . \vec{x}_j + 1)^n$$.

In general, the more complex the problem, the higher the order of the kernel, which is why for the complex discrimination in the project, you should use a radial-basis-function kernel.

----------------

First build the 8 input row vectors using a list comprehension.

In [ ]:
X3s = cvx.matrix([[i,j,k] for i in [0,1] for j in [0,1] for k in [0,1]]).T

In [ ]:
T3_xor = cvx.matrix([-1,1,1,-1,1,-1,-1,1], tc='d')

In [ ]:
q3s = cvx.matrix(len(T3_xor)*[-1.])

In [ ]:
G3 = cvx.matrix(-np.eye(len(T3_xor)))

In [ ]:
h3 = cvx.matrix(len(T3_xor)*[0.])

In [ ]:
def Kn(xi,xj,n=3):
    """Polynomial kernel: (x.y + 1)^n.  (Default: n=3.)
       Accepts CVXOPT row vectors xi, xj, or matrices of such.  
       Generates the polynomial kernel (xi.xj + 1)**n."""
    a = xi * xj.T
    return (a + 1)**n


In [ ]:
P3_xor = cvx.mul(T3_xor * T3_xor.T, Kn(X3s,X3s))

In [ ]:
r_xor3 = cvx.solvers.qp(P3_xor,q3s,G3,h3,T3_xor.T,bv)

In [ ]:
lambdas_xor3 = cvx.matrix([round(li,6) for li in r_xor3['x']])
print(f"{r_xor3['status']}\n{lambdas_xor3}")

We can make use of the routines already defined for the 2-d XOR to do bias calculation and classification (I was careful to make these routines agnostic about the dimensions of their inputs).

In [ ]:
b_xor3 = makeB(Kn,lambdas_xor3,X3s,T3_xor)
print(b_xor3)

In [ ]:
for x1 in [0,1]:
    for x2 in [0,1]:
        for x3 in [0,1]:
            xvec = [x1,x2,x3]
            print(f"{xvec} --> "
                  f"{kernClassify(xvec, Kn, lambdas_xor3, X3s, T3_xor, b_xor3):+2d}"
                  f"  ({2*(x1 ^ x2 ^ x3)-1:+2d})")

------------------------------

## More complex inputs.

Let's consider the type of input specified in the project.  Here we have data consisting of $(x,y)$ points with an associated classification, all contained in simple text files, one for training and one for testing. 

There is an example of such a file in this folder (it is the same format as the files for the project, but does not contain the same data).

Load it in using the useful Numpy utility function "<tt>np.<font color="#4050c0">loadtxt</font></tt>".  This reads arrays from text files.  This text file has three numbers per line (a pair of $(x,y)$ co-ordinate values and a class label), and  $100$ lines, so we get a $100 \times 3$ NumPy array.

In [ ]:
dataset = np.loadtxt("sample-dataset.txt", dtype='float')
print(f"Dataset is of shape {dataset.shape[0]} x {dataset.shape[1]}.")

Split out the training inputs and training labels into separte arrays using NumPy array indexing.

In [ ]:
points, labels = dataset[:,:2], dataset[:,2]

In [ ]:
points

In [ ]:
labels

As you can see, the "<tt>points</tt>" array looks like a matrix of row vectors, one per training input.
The "<tt>labels</tt>" array is just a NumPy vector.

------

It can be interesting to plot the data.  Routine "<tt>np.<font color="#4050c0">scatter</font></tt>" from 
"<tt>matplotlib.<font color="#4050c0">pyplot</font></tt>" can do this in a single statement.  Note that it expects 1-d (vector) inputs for the $x$ and $y$ co-ordinate arrays.  Passing the "<tt>labels</tt>" array via the "<tt>c=</tt>" keyword uses the labels to control the colours of the markers.  Here I'm using a "blus-white-red" colourmap ("<tt>bwr</tt>"), so points with labels $= -1$ will be blue, and points with labels $= +1$ will be red.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(points.T[0], points.T[1], c=labels.T, cmap='bwr')
plt.grid()
plt.axis('equal');

---------------------
### Constructing the SVM classifier

Basically just the same thing again.  But this time $n = 3$ won't work. We'll need a higher-order polynomial kernel, $n = 5$, which we have to find by trial and error.  

This is a good example of a problem where an RBF kernel will work well, with no issues about convergence.

In [ ]:
N = len(labels)

# CVXOPT inputs to qp.
qn = cvx.matrix(-np.ones(N))
Gn = cvx.matrix(-np.eye(N))
hn = cvx.matrix(np.zeros(N))
Xn = cvx.matrix(points)
tn = cvx.matrix(labels)

Kn5 = lambda x, y: Kn(x,y,5)       # Define an n=5 polynomial kernel.

Pn_poly5 = cvx.mul(tn*tn.T, Kn5(Xn,Xn))  # Poly kernel, n = 5.

In [ ]:
r_n_poly5 = cvx.solvers.qp(Pn_poly5,qn,Gn,hn,tn.T,bv)

In [ ]:
print(r_n_poly5['status'])
lambdas_n_poly5 = cvx.matrix([round(li,6) for li in r_n_poly5['x']])

How many suport vectors are there?  These correspond to non-zero Lagrange multipliers.  

In [ ]:
print(np.sum((np.array(lambdas_n_poly5) > 1e-10)))

In [ ]:
print(np.array(lambdas_n_poly5).max())

In [ ]:
plt.bar(np.arange(1,101),lambdas_n_poly5)

In [ ]:
bn5 = makeB(Kn5,lambdas_n_poly5,Xn,tn)

In [ ]:
bn5

In [ ]:
misclass = 0
for xvec,lab in zip(points,labels):
    op = kernClassify(xvec, Kn5, lambdas_n_poly5, Xn, tn, bn5)
    if op != lab: 
        misclass += 1
        print(f"[{xvec[0]:7.4f},{xvec[1]:7.4f}] --> {op:+2d} ({lab:+2.0f})")

print(f"\n{misclass}/{len(labels)} misclassifications")


Great, the SVM has learned the input vectors perfectly using a polynomial kernel with $n = 5$.

----------------------------------
Let's try the same thing with a radial basis function (RBF) kernel.

This routine makes use of the handy routine "<tt>scipy.<font color="#4050c0">spatial.distance.cdist</font></tt>" from the SciPy package to calculate all the squared differences $||\vec{x}_i - \vec{x}_j||^2$ between sets of vectors in a single statement.  

In [ ]:
import scipy.spatial

def Krbf(x,y,s2):
    """RBF kernel on two CVXOPT row vectors, or matrices. s2 is the RBF variance parameter."""
    return cvx.matrix(np.exp(-scipy.spatial.distance.cdist(x,y,metric='sqeuclidean')/(2*s2)))


In [ ]:
S2 = 0.2   # Variance for RBF kernel.

Pn_rbf = cvx.mul(tn*tn.T, Krbf(Xn,Xn,S2))


In [ ]:
r_n_rbf = cvx.solvers.qp(Pn_rbf,qn,Gn,hn,tn.T,bv)

In [ ]:
lambdas_rbf = cvx.matrix([round(li,6) for li in r_n_rbf['x']])

In [ ]:
Krbf_02 = lambda x,y: Krbf(x,y,S2)     # RBF kernel with s2 = 0.2

b_rbf = makeB(Krbf_02,lambdas_rbf,Xn,tn)

In [ ]:
np.set_printoptions(precision=6,suppress=True)
print(np.array(lambdas_rbf).ravel())

print('\n', b_rbf)

In [ ]:
misclass = 0
for xvec,lab in zip(points,labels):
    op = kernClassify(xvec, Krbf_02, lambdas_rbf, Xn, tn, b_rbf)
    if op != lab: 
        misclass += 1
        print(f"[{xvec[0]:7.4f},{xvec[1]:7.4f}] --> {op:+2d} ({lab:+2.0f})")

print(f"\n{misclass}/{len(labels)} misclassifications")


So with an RBF kernel (with $\sigma^2 = 0.2$), we get perfect classification for this problem (on the training set).  Of course, this may not be what you want, with such an overlap of training points perfect classification suggests that the solution is <em>Overfitted</em>.

How many support vectors are there in the RBF kernel?  

In [ ]:
np.sum(np.array(lambdas_rbf)>1e-10)

----------------------------------

Here we plot the decision boundaries and margins for the poly kernel solution and the RBF kernel solution.

I'm using code from a library that's available in this folder as "plotContours.py".  It's based around routines "<tt><font color="#5070b0">contour</font></tt>" and "<tt><font color="#5070b0">contourf</font></tt>" from "<tt>matplotlib.<font color="#5070b0">pyplot</font></tt>".

In [ ]:
from plotContours import plotContours   

In [ ]:
plt.figure(figsize=(16,8))

plt.subplot(1,2,1)
plotContours(Xn, tn, lambdas_n_poly5, bn5, Kn5,
             minRange=-1.5, maxRange=2.5, step=0.01, filled=True)
plt.title("Polynomial kernel with $n=5$")

plt.subplot(1,2,2)
plotContours(Xn, tn, lambdas_rbf, b_rbf, Krbf_02,
             minRange=-1.5, maxRange=2.5, filled=True)
plt.title("RBF kernel with $\\sigma^2=0.2$");


You can see that the polynomial kernel ($n=5$) has <font color="red"><em>very narrow</em></font> margins, which is generally bad for generalisation.  The RBF kernel ($\sigma^2=0.2$) has somewhat wider margins, so should prove to be a better generaliser (should be better with <em>unseen</em> inputs).
